# 1、HumanInTheLoopMiddleware中间件

## 1.1 举例的过程1：工具调用的中断

In [2]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=CLOSEAI_API_KEY,
    base_url=CLOSEAI_BASE_URL
)

In [9]:
from langchain_core.runnables import RunnableConfig
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage
from langchain.tools import tool
from langgraph.types import Command
from rich import print as rprint
from langchain.agents.middleware.human_in_the_loop import InterruptOnConfig

@tool
def get_weather(city: str, is_forcast: bool = False) -> str:
    """
    查询指定城市天气

    Args:
        city: 城市名称
        is_forcast: 是否包含明日天气预报？
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天下雨"
    return res


@tool
def get_news() -> str:
    """
    查询当日新闻
    """
    return "中方三艘油轮通过霍尔木兹海峡"


@tool
def read_email_tool(email_id: str) -> str:
    """通过邮件ID读取内容的伪函数"""
    return f"邮件ID：{email_id}\n是空的"


@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """发送邮件伪函数"""
    print(">>> 真的执行发送邮件工具了")
    return f"发送给 {recipient} 的邮件标题是：{subject}，内容：{body}"


# agent = create_agent(
#     model=model,
#     tools=[get_weather, get_news, read_email_tool, send_email_tool],
#     checkpointer=InMemorySaver(), # 启动短期记忆
#     middleware=[
#         HumanInTheLoopMiddleware(
#             interrupt_on={
#                 "get_weather": True,
#                 "get_news": True,
#                 "read_email_tool": False,
#                 "send_email_tool": {
#                     "allowed_decisions": ["approve", "edit", "reject"],
#                     "description": "发送邮件前需要人工审批"
#                 },
#             },
#             description_prefix="中断啦！！"
#         ),
#     ]
# )



email_config: InterruptOnConfig = {
    "allowed_decisions": ["approve", "edit", "reject"],
    "description": "发送邮件前需要人工审批"
}

interrupt_config = {
    "get_weather": True,
    "get_news": True,
    "read_email_tool": False,
    "send_email_tool": email_config,
}

agent = create_agent(
    model=model,
    tools=[get_weather, get_news, read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on=interrupt_config,
            description_prefix="中断啦！！"
        )
    ]
)

config = RunnableConfig(
    configurable={"thread_id": "1"}
)

response = agent.invoke({
    "messages": [HumanMessage(content="请帮我查询今天北京的天气"
                                      "查询今日新闻"
                                      "查看ID为 'sk2131421' 的邮件内容，"
                                      "向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'"
                                      "同时做这四件事")]
},
    config=config
)


rprint(response)

{
    'messages': [
        HumanMessage(
            content="请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 
的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事",
            additional_kwargs={},
            response_metadata={},
            id='b60b446c-8830-4a4e-8835-666ff317ea4d'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 99,
                    'prompt_tokens': 281,
                    'total_tokens': 380,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
                    'latency_checkpoint': {
                        'engine_tbt_ms': 3,
                        'engine_ttft_ms': 28,
                        'engine_ttlt_ms': 319,
                        'pre_inference_ms': 156,
                        'service_tbt_ms': 3,
                        'service_ttft_ms': 429,
                        'service_ttlt_ms': 692,
                        'user_visible_ttft_ms': 273
                    }
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5.4-mini-2026-03-17',
                'system_fingerprint': None,
                'id': 'chatcmpl-EgWUxS9loZEBmeRm4yMmTw7pr4gf1',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1a703-c9fd-715e-bbf9-0cfc2d250fb5-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '北京', 'is_forcast': False},
                    'id': 'call_SUVTnyNgDYYh2AgPzPIaZv7w',
                    'type': 'tool_call'
                },
                {'name': 'get_news', 'args': {}, 'id': 'call_I2lRXmmO2B6xl7eicUrO7TBV', 'type': 'tool_call'},
                {
                    'name': 'read_email_tool',
                    'args': {'email_id': 'sk2131421'},
                    'id': 'call_vZuNoBsyfehxP656zoB71exa',
                    'type': 'tool_call'
                },
                {
                    'name': 'send_email_tool',
                    'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
                    'id': 'call_pKOMsgTgnmW53O1C8GjSOU09',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 281,
                'output_tokens': 99,
                'total_tokens': 380,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        )
    ],
    '__interrupt__': [
        Interrupt(
            value={
                'action_requests': [
                    {
                        'name': 'get_weather',
                        'args': {'city': '北京', 'is_forcast': False},
                        'description': "中断啦！！\n\nTool: get_weather\nArgs: {'city': '北京', 'is_forcast': 
False}"
                    },
                    {'name': 'get_news', 'args': {}, 'description': '中断啦！！\n\nTool: get_news\nArgs: {}'},
                    {
                        'name': 'send_email_tool',
                        'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
                        'description': '发送邮件前需要人工审批'
                    }
                ],
                'review_configs': [
                    {'action_name': 'get_weather', 'allowed_decisions': ['approve', 'edit', 'reject']},
                    {'action_name': 'get_

## 举例的过程2：指明工具调用请求决策



In [11]:


weather_decision = {
    "type" : "edit",
    "edited_action" : {
        "name" : "get_weather",
        "args" : {"city" : "上海市","is_forcast" : True},
    }
}


news_decision = {
    "type" : "approve"
}


send_email_decision = {
    "type" : "approve"
}



decisions = {
    "decisions" : []
}

interrupts = response.get("__interrupt__",[])
action_requests = interrupts[0].value["action_requests"]


for action_request in action_requests:
    if action_request["name"] == "get_weather":
        decisions["decisions"].append(weather_decision)
    if action_request["name"] == "get_news":
        decisions["decisions"].append(news_decision)
    if action_request["name"] == "send_email_tool":
        decisions["decisions"].append(send_email_decision)

if interrupts :
    resumed_response = agent.invoke(
        Command(resume=decisions),
        config = config
    )

    for msg in resumed_response["messages"]:
        msg.pretty_print()

>>> 真的执行发送邮件工具了
================================ Human Message =================================

请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_SUVTnyNgDYYh2AgPzPIaZv7w)
 Call ID: call_SUVTnyNgDYYh2AgPzPIaZv7w
  Args:
    city: 上海市
    is_forcast: True
  get_news (call_I2lRXmmO2B6xl7eicUrO7TBV)
 Call ID: call_I2lRXmmO2B6xl7eicUrO7TBV
  Args:
  read_email_tool (call_vZuNoBsyfehxP656zoB71exa)
 Call ID: call_vZuNoBsyfehxP656zoB71exa
  Args:
    email_id: sk2131421
  send_email_tool (call_pKOMsgTgnmW53O1C8GjSOU09)
 Call ID: call_pKOMsgTgnmW53O1C8GjSOU09
  Args:
    recipient: 15641685664@qq.com
    subject: 哈哈哈
    body: 你好啊
================================= Tool Message =================================
Name: get_weather

上海市今天天气不错
明天下雨
================================= Tool Message =================================
Name: get_news

